# 02. Деревья решений

Код восстановлен по PDF-отчёту. Результаты будут рассчитаны при выполнении; точное совпадение с отчётом не гарантируется. Запускайте ячейки сверху вниз.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src' / 'project.py').exists():
    raise RuntimeError('Запустите Jupyter из корня проекта, где находятся src/ и README.md')
sys.path.insert(0, str(ROOT / 'src'))
from project import *
frame, (X_train, X_test, y_train, y_test) = load_split()

Здесь исследуем деревья на train и CV. Независимый test оценивается только в третьем ноутбуке после выбора параметров.

In [ ]:
models = trees()
rows = []
for name, model in models.items():
    result = cross_validate(model, X_train, y_train, cv=cv(), scoring=['roc_auc','f1','recall'], n_jobs=-1)
    model.fit(X_train, y_train)
    rows.append({'model': name, 'Train AUC': roc_auc_score(y_train, model.predict_proba(X_train)[:,1]), 'CV AUC': result['test_roc_auc'].mean(), 'CV std': result['test_roc_auc'].std()})
display(pd.DataFrame(rows))

In [ ]:
model = models['Tree gini d=3']
plt.figure(figsize=(23, 10))
plot_tree(model.named_steps['model'], feature_names=model.named_steps['prep'].get_feature_names_out(), class_names=['no','yes'], filled=True, rounded=True, fontsize=8)
plt.tight_layout(); plt.show()

Левая ветвь означает выполнение условия ≤, правая — >. class_weight влияет на распределения классов и критерий. Выигрыш в ячейке ниже — уменьшение Джини, а не энтропийный information gain.

In [ ]:
tree = model.named_steps['model'].tree_
l, r = tree.children_left[0], tree.children_right[0]
w = tree.weighted_n_node_samples
gain = tree.impurity[0] - (w[l]*tree.impurity[l] + w[r]*tree.impurity[r])/w[0]
print('Признак:', model.named_steps['prep'].get_feature_names_out()[tree.feature[0]])
print('Порог:', tree.threshold[0], 'Джини:', tree.impurity[0], 'Выигрыш:', gain)

Сравните Train AUC и CV AUC: большой разрыв у глубокого дерева указывает на переобучение. Итоговые матрицы ошибок и метрики test сохраняет третий ноутбук.